# 00 — Preprocessing: from the raw project exports to the modelling tables

Every table the modelling notebooks (`01`–`07`) consume is produced **here**, from data in the
project root. Nothing downstream touches raw satellite exports.

## Provenance

| output (`Claude/data_cache/`) | built from (project root) | used by |
|---|---|---|
| `features_20d_leakfree.parquet` | `Data_Windowed_{year}_20d/` + `Ertragsdaten_USA.csv` | all notebooks — **the model input** |
| `counts_20d.parquet` | `data_sat_soil_20d_chunks/` | coverage-% pooling bias |
| `cohort_10d_2025.txt` | `Data_Windowed_2025_10d/` + `Ertragsdaten_USA.csv` | the headline evaluation cohort |
| `soil_means_20d.parquet` | `Data_Astatic_wheat_soil_cells/` | notebook `02` only (the headline model uses no soil) |

Upstream of this notebook, the satellite exports themselves are produced by the Google Earth
Engine scripts in the project root (`GEE-On_server_HLSL.py` for Landsat-named years,
`GEE-On_server_HLSS.py` for Sentinel-named years), which apply the USDA CDL wheat mask and the
Fmask cloud screen, and reduce each county-window to per-band statistics. Those exports are the
starting point here and are not regenerated.

Set `REBUILD = True` in the first code cell to rebuild from raw instead of loading the caches
(feature table ~2 min; counts and soil ~10 min each, they scan thousands of per-cell files).

In [6]:
# Setup. Resolves the repo root by walking up from the working directory.
import os, glob, re, time, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

_p = Path.cwd()
while not (_p / "Ertragsdaten_USA.csv").exists():
    assert _p.parent != _p, "repo root not found"
    _p = _p.parent
ROOT = str(_p)
DATA = ROOT + r"\Claude\data_cache"          # derived tables (this notebook writes them)
os.makedirs(DATA, exist_ok=True)
WIN, HIST_K = 7, 5                            # 20-day windows per season; history length
print("repo root :", ROOT)
print("cache dir :", DATA)

repo root : <repository>
cache dir : <repository>\Claude\data_cache


In [7]:
HERE_CACHE = ROOT + r"\Claude\final_results_claude\cache"
# What exists in the project root (the inputs this notebook consumes)
rows = []
for pat, what in [("Data_Windowed_*_20d", "county-mean 20-day windows (model input)"),
                  ("Data_Windowed_*_10d", "county-mean 10-day windows (cohort definition only)"),
                  ("data_sat_soil_20d_chunks", "per-cell sat+soil chunks (pixel counts)"),
                  ("Data_Astatic_wheat_soil_cells", "per-cell static SoilGrids (notebook 02 only)"),
                  ("Data_Fused_Zero_Shot_20d", "zero-shot states (notebook 07)")]:
    for d in sorted(glob.glob(ROOT + "\\" + pat)):
        rows.append(dict(folder=os.path.basename(d), files=len(glob.glob(d + r"\*.csv")), contents=what))
inv = pd.DataFrame(rows)
print(f"Ertragsdaten_USA.csv : {os.path.getsize(ROOT + chr(92) + 'Ertragsdaten_USA.csv')/1e6:.1f} MB (yield labels 2007-2025)")
display(inv)

Ertragsdaten_USA.csv : 0.9 MB (yield labels 2007-2025)


,folder,files,contents
0,Data_Windowed_2016_20d,499,county-mean 20-day windows (model input)
1,Data_Windowed_2017_20d,508,county-mean 20-day windows (model input)
2,Data_Windowed_2018_20d,422,county-mean 20-day windows (model input)
3,Data_Windowed_2019_20d,503,county-mean 20-day windows (model input)
4,Data_Windowed_2020_20d,504,county-mean 20-day windows (model input)
5,Data_Windowed_2021_20d,307,county-mean 20-day windows (model input)
6,Data_Windowed_2022_20d,438,county-mean 20-day windows (model input)
7,Data_Windowed_2023_20d,513,county-mean 20-day windows (model input)
8,Data_Windowed_2025_20d,426,county-mean 20-day windows (model input)
9,Data_Windowed_2016_10d,252,county-mean 10-day windows (cohort definition ...


## 1. Model input table
One row per county-year: 7 windows × 104 band statistics, the yield label, and a leak-free 5-year yield history.

In [8]:
# ---------------------------------------------------------------- features_20d_leakfree.parquet
# One row per county-year. Flattens each Harvester_{year}_{state}_{county}.csv (7 rows x 104
# band statistics) into w00_..w06_ columns, and joins the yield label plus a LEAK-FREE 5-year
# history (mean of the county's yields in the 5 strictly-prior years; NaN if none, imputed later
# from the training split only). County-years without a yield label are dropped.
# Source: Data_Windowed_{year}_20d/ + Ertragsdaten_USA.csv   [orig: experiments/build_features_20d.py]
FEAT = DATA + r"\features_20d_leakfree.parquet"
YEARS = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2025]      # 2024: USDA budget cut, 57 labels
META = {"date_start", "date_end", "from_data"}
REBUILD = False                                                     # True = rebuild even if cached

def build_features():
    y = pd.read_csv(ROOT + r"\Ertragsdaten_USA.csv")
    ycols = {int(c.split()[1]): c for c in y.columns if c.startswith("Yield ")}
    look = {}
    for _, r in y.iterrows():
        look[str(r["id_combo"])] = {yr: float(r[c]) for yr, c in ycols.items() if pd.notna(r[c])}
    rows, stat_cols, skipped = [], None, 0
    for year in YEARS:
        for f in glob.glob(ROOT + rf"\Data_Windowed_{year}_20d\Harvester_*.csv"):
            b = os.path.basename(f)[:-4].split("_"); cid = f"{b[2]}_{b[3]}"
            t = pd.read_csv(f)
            if len(t) != WIN:                        # incomplete season -> unusable
                skipped += 1; continue
            if stat_cols is None:
                stat_cols = [c for c in t.columns if c not in META]
            d = look.get(cid, {})
            if year not in d:                        # no yield label -> unusable
                skipped += 1; continue
            hv = [d[year - k] for k in range(1, HIST_K + 1) if (year - k) in d]   # strictly prior
            rec = {"county_id": cid, "year": year, "yield": d[year],
                   "yield_5yr_avg": float(np.mean(hv)) if hv else np.nan}
            a = t[stat_cols].to_numpy(np.float32)    # file row order == window order
            for w in range(WIN):
                for j, c in enumerate(stat_cols):
                    rec[f"w{w:02d}_{c}"] = a[w, j]
            rows.append(rec)
        print(f"  {year}: {len(rows)} rows so far", flush=True)
    out = pd.DataFrame(rows)
    out.to_parquet(FEAT, index=False)
    print(f"skipped (no label / incomplete season): {skipped}")
    return out

if REBUILD or not os.path.exists(FEAT):
    feat = build_features()
else:
    feat = pd.read_parquet(FEAT); print("loaded cached", os.path.basename(FEAT))
feat["county_id"] = feat.county_id.astype(str)
print(f"\nfeatures_20d_leakfree: {feat.shape[0]} county-years x {feat.shape[1]} cols")
print(feat.year.value_counts().sort_index().to_string())
print(f"\nmissing yield_5yr_avg (no prior years): {feat.yield_5yr_avg.isna().sum()}"
      f"  -> imputed from the TRAIN split only, inside each model script")

loaded cached features_20d_leakfree.parquet

features_20d_leakfree: 4120 county-years x 732 cols
year
2016    499
2017    508
2018    422
2019    503
2020    504
2021    307
2022    438
2023    513
2025    426

missing yield_5yr_avg (no prior years): 48  -> imputed from the TRAIN split only, inside each model script


## 2. Pixel counts (satellite coverage)
Only `wheat_pix` and the per-window `satc_*` counts are used, as the coverage-% bias on attention pooling.

In [9]:
# ---------------------------------------------------------------- counts_20d / counts_10d
# Per county-year: how many wheat pixels back the county mean (denominator), and how many the
# satellite observed in each window (numerator). Only `wheat_pix` and `satc_w*` are used by the
# model, as the coverage-% bias on attention pooling.
# Sources: Data_Astatic_wheat_soil_cells/*.csv (denominator), Data_Fused_<year>/*.csv (numerator)
CNT20 = DATA + r"\counts_20d.parquet"
CNT10 = DATA + r"\counts_10d.parquet"
SOILCOL = "awc_0_30_mean_1km"                  # named without the "soil_" prefix in these files
CELL_PAT = re.compile(r"county_(\d+)_(\d+)_(\d{4})_static")
SEASON_START = pd.Timestamp(2000, 4, 7)


def build_static_pixels():
    """Denominator: wheat pixels per county-year, from the static wheat and soil cells."""
    rows = []
    for f in sorted(glob.glob(ROOT + r"\Data_Astatic_wheat_soil_cells\*.csv")):
        m = CELL_PAT.match(os.path.basename(f))
        if not m:
            continue
        d = pd.read_csv(f, usecols=["static_wheat_pixels", "static_wheat_pixels_incounty", SOILCOL])
        rows.append({"county_id": f"{m.group(1)}_{m.group(2)}", "year": int(m.group(3)),
                     "n_cells": len(d),
                     "wheat_pix": float(d.static_wheat_pixels.sum()),
                     "wheat_pix_incty": float(d.static_wheat_pixels_incounty.sum()),
                     "soil_valid_frac": float(d[SOILCOL].notna().mean())})
    return pd.DataFrame(rows).drop_duplicates(["county_id", "year"])


def build_observed_pixels():
    """Numerator: observed pixels per window, both window lengths from one pass over the files."""
    from concurrent.futures import ThreadPoolExecutor

    def one_file(args):
        year, f = args
        parts = os.path.basename(f)[:-4].split("_")
        d = pd.read_csv(f, usecols=["date", "B3_count"], dtype={"date": str})
        day_month = d.date.str.split(".", expand=True)
        dates = pd.to_datetime(dict(year=2000, month=day_month[1].astype(int),day=day_month[0].astype(int)))
        days = (dates - SEASON_START).dt.days
        records = []
        for win_days, n_windows in [(20, WIN), (10, 2 * WIN)]:
            window_of_row = days // win_days
            rec = {"county_id": f"{parts[2]}_{parts[3]}", "year": year}
            for window in range(n_windows):
                rec[f"satc_w{window:02d}"] = float(d.B3_count[window_of_row == window].sum())
            records.append(rec)
        return records

    jobs = [(y, f) for y in YEARS for f in sorted(glob.glob(ROOT + rf"\Data_Fused_{y}\Harvester_*.csv"))]
    print(f"{len(jobs)} fused county-year files (slow: ~20 min)", flush=True)
    with ThreadPoolExecutor(max_workers=8) as pool:
        results = list(pool.map(one_file, jobs))
    return pd.DataFrame([r[0] for r in results]), pd.DataFrame([r[1] for r in results])


if REBUILD or not os.path.exists(CNT20) or not os.path.exists(CNT10):
    static_pixels = build_static_pixels()
    observed_20d, observed_10d = build_observed_pixels()
    cnt20 = static_pixels.merge(observed_20d, on=["county_id", "year"], how="left")
    cnt10 = static_pixels.merge(observed_10d, on=["county_id", "year"], how="left")
    cnt20.to_parquet(CNT20, index=False)
    cnt10.to_parquet(CNT10, index=False)
else:
    cnt20 = pd.read_parquet(CNT20); cnt10 = pd.read_parquet(CNT10)
    print("loaded cached", os.path.basename(CNT20), "and", os.path.basename(CNT10))

for name, cnt in [("counts_20d", cnt20), ("counts_10d", cnt10)]:
    cnt["county_id"] = cnt.county_id.astype(str)
    print(f"{name}: {cnt.shape[0]} county-years")
    cov = feat[["county_id", "year"]].merge(cnt[["county_id", "year"]], on=["county_id", "year"],
                                            how="left", indicator=True)
    print(f"  feature rows WITHOUT counts (coverage-% median-imputed): {(cov._merge=='left_only').sum()} / {len(feat)}")

4763 fused county-year files (slow: ~20 min)
counts_20d: 5387 county-years
  feature rows WITHOUT counts (coverage-% median-imputed): 0 / 4120
counts_10d: 5387 county-years
  feature rows WITHOUT counts (coverage-% median-imputed): 0 / 4120


## 3. The evaluation cohorts
Two nested test sets from the held-out 2025 season: the **headline** cohort of counties with complete 10-day coverage (well-observed), and the **full** 2025 population. The model always uses the 20-day features; the 10-day series only decides which counties are well observed.

In [10]:
# ---------------------------------------------------------------- the evaluation cohort
# Results are reported on two nested test sets, both from the held-out 2025 season:
#   HEADLINE  county-years with COMPLETE 10-day satellite coverage (all 14 windows) and a
#             yield label -- the data-rich subset, i.e. counties observed often enough that
#             cloud gaps do not dominate the season.
#   FULL      every 2025 county-year in the feature table (includes counties whose 10-day
#             coverage is incomplete, which are systematically harder).
# The model itself always uses the 20-day features; the 10-day series is used ONLY to define
# which counties are well observed. Both sets include all states, Washington included.
cty_in_2025 = set(pd.read_csv(ROOT + r"\Ertragsdaten_USA.csv")
          .pipe(lambda d: d.loc[d["Yield 2025"].notna(), "id_combo"]).astype(str))
complete10, short = set(), 0
for p in glob.glob(ROOT + r"\Data_Windowed_2025_10d\Harvester_2025_*.csv"):
    b = os.path.basename(p)[:-4].split("_")
    cid = f"{b[2]}_{b[3]}"
    read_file_check_num_windows = pd.read_csv(p)
    if len(read_file_check_num_windows) == 14:
        complete10.add(cid)
    else: 
        short += 1
cohort = complete10 & cty_in_2025
cty_with_20d_data = set(feat.loc[feat.year == 2025, "county_id"])
print(f"2025 10-day files: {len(complete10)+short} (complete 14-window: {len(complete10)}, incomplete: {short})")
print(f"  ... with a 2025 yield label            : {len(cohort)}")
print(f"  ... present in the 20-day feature table: {len(cohort & cty_with_20d_data)}")
cohort &= cty_with_20d_data
os.makedirs(HERE_CACHE, exist_ok=True)
open(HERE_CACHE + r"\cohort_10d_2025.txt", "w").write("\n".join(sorted(cohort)))
print(f"\nHEADLINE cohort (10-day complete): {len(cohort)} county-years -> cohort_10d_2025.txt")
print(f"FULL 2025 population             : {len(cty_with_20d_data)} county-years")
print(f"  (the {len(cty_with_20d_data)-len(cohort)} counties outside the headline cohort have incomplete"
      f" 10-day coverage and are harder to predict)")

2025 10-day files: 282 (complete 14-window: 282, incomplete: 0)
  ... with a 2025 yield label            : 282
  ... present in the 20-day feature table: 282

HEADLINE cohort (10-day complete): 282 county-years -> cohort_10d_2025.txt
FULL 2025 population             : 426 county-years
  (the 144 counties outside the headline cohort have incomplete 10-day coverage and are harder to predict)


## 4. County-mean soil *(notebook 02 only)*
The headline model uses no soil; this table exists to support the ablations that establish that.

In [11]:
# ---------------------------------------------------------------- soil_means_20d.parquet
# ONLY needed by notebook 02 (the soil-mechanism ablations). The headline model uses NO soil.
# County-mean of the 18 SoilGrids properties over the county's wheat cells.
# Source: Data_Astatic_wheat_soil_cells/   [orig: county_means/build_soilmeans_astatic.py]
SOIL = DATA + r"\soil_means_20d.parquet"
PAT = re.compile(r"county_(\d+)_(\d+)_(\d{4})_static")

def build_soil():
    files = sorted(glob.glob(ROOT + r"\Data_Astatic_wheat_soil_cells\*.csv"))
    val_cols = [c for c in pd.read_csv(files[0], nrows=1).columns
                if c.endswith("_mean_1km") and "pixel_count" not in c]
    print(f"{len(files)} files | {len(val_cols)} soil value columns", flush=True)
    rows, bad, t0 = [], 0, time.time()
    for i, f in enumerate(files):
        m = PAT.match(os.path.basename(f))
        if not m:
            bad += 1; continue
        try:
            d = pd.read_csv(f, usecols=val_cols)
        except Exception:
            bad += 1; continue
        rec = {"county_id": f"{m.group(1)}_{m.group(2)}", "year": int(m.group(3))}
        for c in val_cols:
            rec["soil_" + c] = float(d[c].mean())
        rows.append(rec)
        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(files)} ({time.time()-t0:.0f}s)", flush=True)
    out = pd.DataFrame(rows).drop_duplicates(["county_id", "year"])
    out.to_parquet(SOIL, index=False); print(f"skipped {bad}")
    return out

if REBUILD or not os.path.exists(SOIL):
    soil = build_soil()
else:
    soil = pd.read_parquet(SOIL); print("loaded cached", os.path.basename(SOIL))
soil["county_id"] = soil.county_id.astype(str)
nsoil = sum(c.startswith("soil_") for c in soil.columns)
print(f"soil_means_20d: {soil.shape[0]} county-years x {nsoil} soil properties")
k = lambda d: set(zip(d.county_id, d.year))
print(f"feature rows missing soil: {len(k(feat) - k(soil))} / {len(feat)}")

5387 files | 18 soil value columns
  1000/5387 (2s)
  2000/5387 (6s)
  3000/5387 (8s)
  4000/5387 (13s)
  5000/5387 (17s)
skipped 0
soil_means_20d: 5387 county-years x 18 soil properties
feature rows missing soil: 0 / 4120


## 5. Integrity check

In [12]:
# ---------------------------------------------------------------- integrity check
# Confirms the tables the modelling notebooks consume are exactly what this notebook describes.
ok = True
def check(label, got, want):
    global ok
    good = got == want; ok &= good
    print(f"  [{'OK ' if good else 'FAIL'}] {label}: {got}" + ("" if good else f"  (expected {want})"))

print("integrity:")
check("feature county-years", len(feat), 4120)
check("feature columns", feat.shape[1], 732)
check("2025 held-out county-years (full)", int((feat.year == 2025).sum()), 426)
check("headline cohort (10-day complete)", len(cohort), 282)
check("soil properties", nsoil, 18)
check("years covered", len(feat.year.unique()), 9)
print("\nALL CHECKS PASSED" if ok else "\nMISMATCH - see above")
print("\nDownstream: notebooks 01-07 read these tables; none of them re-derive raw data.")

integrity:
  [OK ] feature county-years: 4120
  [OK ] feature columns: 732
  [OK ] 2025 held-out county-years (full): 426
  [OK ] headline cohort (10-day complete): 282
  [OK ] soil properties: 18
  [OK ] years covered: 9

ALL CHECKS PASSED

Downstream: notebooks 01-07 read these tables; none of them re-derive raw data.


## Summary

Raw exports → four derived tables → notebooks `01`–`07`. The complete chain:

1. **Google Earth Engine** (`GEE-On_server_HLS*.py`, project root) — CDL wheat mask, Fmask cloud
   screen, per-county-window band statistics → `Data_Windowed_*`
2. **This notebook** — flatten to one row per county-year, join leak-free yield history, derive
   pixel counts, freeze the evaluation cohort → `Claude/data_cache/`
3. **Notebooks 01–07** — feature construction (lean tokens), model training, evaluation

The only step not reproducible from this repository is the Earth Engine export itself, which
requires a Google Earth Engine account and runs server-side.